# Gemma 4 12B - Colab server (Gradio UI + OpenAI API via ngrok)
Run **Runtime > Run all**. The last cell prints the public URL.

In [1]:
!nvidia-smi

Sat Nov 14 12:25:08 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   77C    P0             32W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
# 1. Mount Drive (robust) + config
import os
from google.colab import drive

if not os.path.isdir("/content/drive/MyDrive"):
    try:
        drive.mount("/content/drive")
    except Exception as e:
        print("Mount failed, retrying with force_remount:", e)
        drive.mount("/content/drive", force_remount=True)

assert os.path.isdir("/content/drive/MyDrive"), "Drive is not mounted - approve the popup and re-run."

BASE_DIR = "/content/drive/MyDrive/LocalLLM"
MODEL_ID = "gemma4-12b-balanced"
MODEL_CONTEXT_TOKENS = 24576

# >>> FILL THESE IN: the Hugging Face repo that hosts your GGUF <<<
HF_REPO_ID  = "OWNER/REPO"
HF_FILENAME = "Gemma4-12B-QAT-Uncensored-HauhauCS-Balanced-Q4_K_M.gguf"

DRIVE_MODEL_DIR = f"{BASE_DIR}/models/gemma4-12b-balanced"
WHEEL_ROOT = f"{BASE_DIR}/wheels"
for d in (DRIVE_MODEL_DIR, WHEEL_ROOT):
    os.makedirs(d, exist_ok=True)
print("Drive OK:", BASE_DIR)

Drive OK: /content/drive/MyDrive/LocalLLM


In [3]:
# 2. Install deps; llama-cpp-python wheel is cached in Drive and reused
import sys, re, glob, subprocess

def sh(cmd):
    return subprocess.run(cmd, shell=True, capture_output=True, text=True)

r = sh(f"{sys.executable} -m pip -q install gradio pyngrok pypdf huggingface_hub nvidia-ml-py")
if r.returncode:
    print(r.stderr[-2000:])

pyv = f"cp{sys.version_info.major}{sys.version_info.minor}"
nvcc = sh("nvcc --version").stdout
m = re.search(r"release (\d+)\.(\d+)", nvcc)
detected = f"cu{m.group(1)}{m.group(2)}" if m else None
print("Python:", pyv, "| CUDA toolkit tag:", detected)

WHEEL_DIR = f"{WHEEL_ROOT}/{pyv}"
os.makedirs(WHEEL_DIR, exist_ok=True)
find = lambda: sorted(glob.glob(f"{WHEEL_DIR}/llama_cpp_python*.whl"))

if not find():
    tags = []
    for t in (detected, "cu130", "cu125", "cu124"):
        if t and t not in tags:
            tags.append(t)
    for tag in tags:
        print("Downloading wheel from index", tag, "...")
        r = sh(f"{sys.executable} -m pip download llama-cpp-python --only-binary=:all: --no-deps "
               f"-d {WHEEL_DIR} --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/{tag}")
        if r.returncode == 0 and find():
            break
        print(r.stderr[-500:])
    if not find():
        raise RuntimeError("No prebuilt CUDA wheel found for this Python/CUDA. Paste the output above and I will adjust.")
else:
    print("Reusing cached wheel from Drive:", os.path.basename(find()[-1]))

r = sh(f"{sys.executable} -m pip -q install --force-reinstall --no-deps {find()[-1]}")
r2 = sh(f"{sys.executable} -m pip -q install numpy typing-extensions diskcache jinja2")
if r.returncode:
    raise RuntimeError(r.stderr[-2000:])

import llama_cpp
print("llama-cpp-python", llama_cpp.__version__, "| GPU offload:", llama_cpp.llama_supports_gpu_offload())

Python: cp313 | CUDA toolkit tag: cu130
Reusing cached wheel from Drive: llama_cpp_python-0.3.36-py3-none-manylinux_2_35_x86_64.whl


ggml_cuda_init: found 1 CUDA devices (Total VRAM: 14912 MiB):
  Device 0: Tesla T4, compute capability 7.5, VMM: yes, VRAM: 14912 MiB


llama-cpp-python 0.3.36 | GPU offload: True


In [4]:
# 3. Model: download once -> persist in Drive -> reuse; load from fast LOCAL copy
import os, time, shutil
from google.colab import userdata

def get_secret(name, default=None):
    """Never crashes: returns default if the secret is missing or notebook access is off."""
    try:
        v = userdata.get(name)
    except Exception as e:
        print(f"[secrets] {name}: {type(e).__name__}: {str(e)[:120]}")
        v = None
    return v or os.environ.get(name) or default

MIN_BYTES = 1_000_000_000
LOCAL_DIR = "/content/models"
os.makedirs(LOCAL_DIR, exist_ok=True)
MODEL_PATH  = f"{LOCAL_DIR}/{HF_FILENAME}"
DRIVE_MODEL = f"{DRIVE_MODEL_DIR}/{HF_FILENAME}"

def _fmt_time(s):
    s = int(s)
    return f"{s // 60}m{s % 60:02d}s"

def safe_copy(src, dst, label="Copying", chunk=16 * 1024 * 1024):
    """Copy with a live progress line; writes to .part first so a half-copied file is never used."""
    total = os.path.getsize(src)
    tmp = dst + ".part"
    if os.path.exists(tmp):
        os.remove(tmp)
    copied, start, last_print = 0, time.time(), 0.0
    with open(src, "rb") as fi, open(tmp, "wb") as fo:
        while True:
            buf = fi.read(chunk)
            if not buf:
                break
            fo.write(buf)
            copied += len(buf)
            now = time.time()
            if now - last_print >= 0.5 or copied == total:
                last_print = now
                elapsed = max(now - start, 1e-6)
                speed = copied / elapsed
                eta = (total - copied) / speed if speed else 0
                pct = copied / total * 100
                bar = "#" * int(pct // 4) + "-" * (25 - int(pct // 4))
                print(f"\r{label}: [{bar}] {pct:5.1f}%  "
                      f"{copied/1024**3:5.2f}/{total/1024**3:.2f} GB  "
                      f"{speed/1024**2:6.1f} MB/s  ETA {_fmt_time(eta)}   ",
                      end="", flush=True)
        fo.flush()
        os.fsync(fo.fileno())
    print()
    if os.path.getsize(tmp) != total:
        os.remove(tmp)
        raise IOError("Copy size mismatch - not keeping partial file.")
    os.replace(tmp, dst)
    print(f"{label}: done in {_fmt_time(time.time() - start)}")

drive_ok = os.path.isfile(DRIVE_MODEL) and os.path.getsize(DRIVE_MODEL) > MIN_BYTES

if os.path.isfile(MODEL_PATH) and os.path.getsize(MODEL_PATH) > MIN_BYTES:
    print("Local copy already present.")
elif drive_ok:
    print("Found model in Drive - copying to local disk (no re-download)...")
    safe_copy(DRIVE_MODEL, MODEL_PATH, label="Drive -> local")
else:
    if HF_REPO_ID.startswith("OWNER/"):
        raise ValueError("Set HF_REPO_ID in cell 1 (or upload the .gguf to: " + DRIVE_MODEL_DIR + ")")
    from huggingface_hub import hf_hub_download
    print("Downloading from Hugging Face (resumes if interrupted)...")
    p = hf_hub_download(repo_id=HF_REPO_ID, filename=HF_FILENAME, local_dir=LOCAL_DIR,
                        token=get_secret("HF_TOKEN"))
    assert os.path.getsize(p) > MIN_BYTES, "Download looks truncated."
    print("Saving to Drive so future sessions skip the download...")
    safe_copy(p, DRIVE_MODEL, label="local -> Drive")
    os.sync()

print("MODEL_PATH:", MODEL_PATH, f"({os.path.getsize(MODEL_PATH)/1024**3:.2f} GB)")

Local copy already present.
MODEL_PATH: /content/models/Gemma4-12B-QAT-Uncensored-HauhauCS-Balanced-Q4_K_M.gguf (6.87 GB)


In [5]:
# 4. Chat DB (created if missing - the old cell crashed when it did not exist)
import sqlite3
CHAT_DB = f"{BASE_DIR}/chat_history.sqlite3"
with sqlite3.connect(CHAT_DB, timeout=30) as con:
    con.execute("""
        CREATE TABLE IF NOT EXISTS messages (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            conversation_id TEXT NOT NULL,
            role TEXT NOT NULL,
            content TEXT NOT NULL,
            created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
        )
    """)
print("Chat database:", CHAT_DB)

Chat database: /content/drive/MyDrive/LocalLLM/chat_history.sqlite3


In [6]:
# 5. Secrets (run every session). Uses get_secret() from cell 3.
ngrok_token  = get_secret("NGROK_AUTHTOKEN")
ngrok_domain = get_secret("NGROK_DOMAIN")
API_KEY      = get_secret("API_KEY", "sk-gemma4-balanced-custom")

missing = [n for n, v in (("NGROK_AUTHTOKEN", ngrok_token), ("NGROK_DOMAIN", ngrok_domain)) if not v]
if missing:
    raise RuntimeError(
        f"Missing/blocked secrets: {missing}. Open the key icon in the left panel, check the names match "
        "exactly (case-sensitive) and that 'Notebook access' is toggled ON for THIS notebook.")

ngrok_domain = ngrok_domain.strip().rstrip("/").removeprefix("https://").removeprefix("http://").split("/")[0]
print("Secrets loaded: NGROK_AUTHTOKEN, NGROK_DOMAIN, API_KEY(" + ("custom" if API_KEY != "sk-gemma4-balanced-custom" else "fallback") + ")")
# In your old ngrok cell, delete the "1. Read Colab Secrets" block and the API_KEY block
# (everything down to "# 2. Configure Ngrok") - these variables are already set here.

Secrets loaded: NGROK_AUTHTOKEN, NGROK_DOMAIN, API_KEY(custom)


In [7]:
import os
import sqlite3
import uuid
import time
import gradio as gr

SYSTEM_PROMPT = (
    "You are a helpful AI assistant. "
    "Give clear, accurate and thoughtful answers."
)

# ---------------- Database helpers ----------------

def get_conversations():
    with sqlite3.connect(CHAT_DB) as con:
        rows = con.execute("""
            SELECT conversation_id, MAX(id) AS last_id
            FROM messages
            GROUP BY conversation_id
            ORDER BY last_id DESC
        """).fetchall()
    return [row[0] for row in rows]


def get_messages(cid):
    if not cid:
        return []

    with sqlite3.connect(CHAT_DB) as con:
        rows = con.execute("""
            SELECT role, content
            FROM messages
            WHERE conversation_id = ?
            ORDER BY id
        """, (cid,)).fetchall()

    return [
        {"role": role, "content": content}
        for role, content in rows
        if role in ("user", "assistant")
    ]


def save_message(cid, role, content):
    with sqlite3.connect(CHAT_DB) as con:
        con.execute("""
            INSERT INTO messages (conversation_id, role, content)
            VALUES (?, ?, ?)
        """, (cid, role, content))


# ---------------- Attachment reader ----------------

def read_uploaded_file(file_path):
    filename = os.path.basename(file_path)
    extension = os.path.splitext(filename)[1].lower()

    if extension == ".txt":
        with open(file_path, "r", encoding="utf-8-sig") as uploaded_file:
            text = uploaded_file.read()
        if not text.strip():
            raise ValueError("The uploaded text file is empty.")
        return text

    if extension == ".pdf":
        from pypdf import PdfReader

        page_texts = []
        for page in PdfReader(file_path).pages:
            page_text = page.extract_text()
            if page_text and page_text.strip():
                page_texts.append(page_text.strip())
        text = "\n\n".join(page_texts)
        if not text:
            raise ValueError(
                "No selectable text was found in the PDF. Scanned PDFs require OCR."
            )
        return text

    raise ValueError("Only .txt and .pdf files are supported.")


# ---------------- Chat actions ----------------

def new_conversation():
    cid = str(uuid.uuid4())[:8]
    choices = get_conversations()

    if cid not in choices:
        choices.insert(0, cid)

    return cid, [], gr.update(choices=choices, value=cid), None


def load_conversation(cid):
    if not cid:
        return "", [], None
    return cid, get_messages(cid), None


def chat(message, cid, file_path=None):
    received_at = time.perf_counter()
    message = (message or "").strip()
    cid = (cid or "").strip()
    print(
        f"[chat] Prompt received ({len(message)} characters); "
        f"conversation={cid or 'new'}, attachment={'yes' if file_path else 'no'}.",
        flush=True,
    )

    if file_path:
        try:
            print("[chat] Reading uploaded attachment...", flush=True)
            file_text = read_uploaded_file(file_path)
        except Exception as e:
            print(f"[chat] Attachment read failed: {type(e).__name__}: {e}", flush=True)
            raise gr.Error(
                f"Unable to read uploaded file ({type(e).__name__}): {e}"
            ) from e

        attachment_prompt = (
            f"Contents of uploaded file ({os.path.basename(file_path)}):\n{file_text}"
        )
        message = "\n\n".join(
            part for part in (message, attachment_prompt) if part
        )
        print(
            f"[chat] Attachment read ({len(file_text)} characters).",
            flush=True,
        )

    if not message:
        print("[chat] Empty prompt; skipping generation.", flush=True)
        yield "", get_messages(cid), cid, gr.update(
            choices=get_conversations(), value=cid or None
        ), None
        return

    if not cid:
        cid = str(uuid.uuid4())[:8]

    save_message(cid, "user", message)
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT}
    ] + get_messages(cid)
    prompt_chars = sum(len(item.get("content") or "") for item in messages)
    print(
        f"[chat] Prompt prepared: {len(messages)} messages, "
        f"{prompt_chars} characters. Formatting with thinking disabled...",
        flush=True,
    )

    choices = get_conversations()
    if cid not in choices:
        choices.insert(0, cid)

    visible_history = get_messages(cid)
    yield (
        "",
        visible_history + [{"role": "assistant", "content": "Generating..."}],
        cid,
        gr.update(choices=choices, value=cid),
        None,
    )

    answer = ""
    first_token_logged = False
    try:
        prompt_text = format_chat_prompt(messages, enable_thinking=False)
        prompt_tokens = llm.tokenize(
            prompt_text.encode("utf-8"), add_bos=False, special=True
        )
        prompt_tokens, max_output_tokens = fit_prompt_to_context(prompt_tokens)

        print(
            f"[chat] Prompt ready ({len(prompt_tokens)} tokens). "
            "Generating response without reasoning...",
            flush=True,
        )
        generation_started = time.perf_counter()
        for chunk in llm.create_completion(
            prompt=prompt_tokens,
            max_tokens=max_output_tokens,
            temperature=0.6,
            top_p=0.9,
            stop=["<end_of_turn>", "<eos>"],
            stream=True,
        ):
            token_text = chunk["choices"][0].get("text", "")
            if not isinstance(token_text, str) or not token_text:
                continue

            if not first_token_logged:
                first_token_logged = True
                print(
                    f"[chat] First response token after "
                    f"{time.perf_counter() - generation_started:.1f}s.",
                    flush=True,
                )

            answer += token_text
            yield (
                "",
                visible_history + [{"role": "assistant", "content": answer}],
                cid,
                gr.update(choices=choices, value=cid),
                None,
            )

        if not answer.strip():
            answer = "(The model returned an empty response.)"
        save_message(cid, "assistant", answer)
        print(
            f"[chat] Generation complete in "
            f"{time.perf_counter() - generation_started:.1f}s "
            f"({len(answer)} response characters).",
            flush=True,
        )
        final_history = get_messages(cid)
    except Exception as e:
        answer = f"Model error: {type(e).__name__}: {e}"
        print(f"[chat] Model generation failed: {answer}", flush=True)
        final_history = visible_history + [{"role": "assistant", "content": answer}]

    print(
        f"[chat] Request finished in {time.perf_counter() - received_at:.1f}s.",
        flush=True,
    )
    yield (
        "",
        final_history,
        cid,
        gr.update(choices=choices, value=cid),
        None,
    )


# ---------------- Gradio interface ----------------

with gr.Blocks(title="My AI Assistant") as demo:
    gr.Markdown("# My AI Assistant")
    gr.Markdown("Gemma 4 12B QAT Balanced · Chat history saved in SQLite")

    with gr.Row():
        cid = gr.Textbox(
            label="Conversation ID",
            placeholder="Create a new chat or select a saved one",
            scale=3
        )
        new_btn = gr.Button("New chat")

    saved_chats = gr.Dropdown(
        label="Saved conversations",
        choices=get_conversations(),
        interactive=True
    )

    chatbot_ui = gr.Chatbot(
    height=500,
    label="Chat"
    )

    message = gr.Textbox(
        label="Your message",
        placeholder="Message your AI...",
        lines=2
    )

    attachment = gr.File(
        label="Attach a .txt or .pdf file",
        file_types=[".txt", ".pdf"],
        type="filepath",
    )

    send_btn = gr.Button("Send", variant="primary")

    new_btn.click(
        new_conversation,
        outputs=[cid, chatbot_ui, saved_chats, attachment]
    )

    saved_chats.change(
        load_conversation,
        inputs=saved_chats,
        outputs=[cid, chatbot_ui, attachment]
    )

    send_btn.click(
        chat,
        inputs=[message, cid, attachment],
        outputs=[message, chatbot_ui, cid, saved_chats, attachment]
    )

    message.submit(
        chat,
        inputs=[message, cid, attachment],
        outputs=[message, chatbot_ui, cid, saved_chats, attachment]
    )

    demo.load(
        lambda: ("", [], None),
        outputs=[cid, chatbot_ui, attachment]
    )

print("Chat functions restored:", callable(chat))
print("Gradio interface restored:", demo is not None)

Chat functions restored: True
Gradio interface restored: True


In [8]:
import gc, llama_cpp
from llama_cpp import Llama

if not llama_cpp.llama_supports_gpu_offload():
    raise RuntimeError("llama-cpp-python was installed without CUDA support. Re-run the install cell.")

# Free any previous model first (re-running this cell otherwise keeps old VRAM allocated)
try:
    del llm
except NameError:
    pass
gc.collect()

def load_llm(n_ctx):
    kwargs = dict(
        model_path=MODEL_PATH,
        n_gpu_layers=-1,
        n_ctx=n_ctx,
        n_batch=512,
        n_ubatch=256,                      # smaller compute buffer
        flash_attn=True,
        type_k=llama_cpp.GGML_TYPE_Q8_0,   # q8 KV cache: roughly half the f16 size
        type_v=llama_cpp.GGML_TYPE_Q8_0,
        verbose=False,
    )
    try:
        return Llama(swa_full=False, **kwargs)   # key fix: sliding-window layers keep only their window
    except TypeError:
        print("This llama-cpp-python has no swa_full option; trying without it.")
        return Llama(**kwargs)

llm, last_err = None, None
for ctx in (MODEL_CONTEXT_TOKENS, 16384, 12288, 8192, 4096):
    try:
        print(f"Trying n_ctx={ctx:,} ...")
        llm = load_llm(ctx)
        MODEL_CONTEXT_TOKENS = ctx
        break
    except Exception as e:
        last_err = e
        print(f"  failed: {type(e).__name__}: {e}")
        gc.collect()

if llm is None:
    raise RuntimeError(f"Could not create a context even at 4,096 tokens: {last_err}")

print(f"Model loaded on GPU with {MODEL_CONTEXT_TOKENS:,}-token context.")

def fit_prompt_to_context(prompt_tokens):
    max_ctx = llm.n_ctx() if callable(llm.n_ctx) else llm.n_ctx
    prompt_limit = max_ctx - 1
    if len(prompt_tokens) > prompt_limit:
        print(f"Trimming prompt from {len(prompt_tokens)} to {prompt_limit} tokens.", flush=True)
        prompt_tokens = prompt_tokens[-prompt_limit:]
    return prompt_tokens, max_ctx - len(prompt_tokens)

Trying n_ctx=24,576 ...
Model loaded on GPU with 24,576-token context.


In [9]:
import os
import time
import socket
import json
import re
import threading
import traceback
from pyngrok import ngrok
from google.colab import userdata
import gradio as gr
from fastapi import FastAPI, Request, Header, HTTPException
from fastapi.responses import JSONResponse, StreamingResponse
from fastapi.middleware.cors import CORSMiddleware
from llama_cpp.llama_chat_format import Jinja2ChatFormatter
import uvicorn

# ---------------- 2. Configure Ngrok & Clean up previous runs ----------------
try:
    ngrok.set_auth_token(ngrok_token)
except Exception as e:
    if "Authtoken already set in configuration file" not in str(e):
        raise

# Terminate old tunnels and servers if re-running cell
try:
    ngrok.kill()
except Exception:
    pass

global uvicorn_server
if "uvicorn_server" in globals() and uvicorn_server is not None:
    uvicorn_server.should_exit = True
    time.sleep(1)

# ---------------- 3. Validate prerequisites ----------------
if "demo" not in globals():
    raise RuntimeError(
        "The variable 'demo' does not exist. Run the Gradio definition cell first."
    )

if "llm" not in globals():
    raise RuntimeError(
        "The variable 'llm' does not exist. Run your model-loading cell first."
    )

if "llm_lock" not in globals():
    llm_lock = threading.Lock()

In [10]:
# ---------------- 4. Define FastAPI with OpenAI Endpoints ----------------
api_app = FastAPI(
    title="Gemma 4 12B OpenAI Compatible Server",
    description="OpenAI-compatible endpoints for Continue (VS Code) & custom code editors",
    version="1.0.0"
)

api_app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

def verify_api_key(authorization: str = None, api_key_header: str = None, x_api_key: str = None, req_api_key: str = None):
    if not API_KEY:
        return True
    token = None
    if authorization:
        token = authorization.replace("Bearer ", "").strip()
    elif api_key_header:
        token = api_key_header.strip()
    elif x_api_key:
        token = x_api_key.strip()
    elif req_api_key:
        token = req_api_key.strip()

    if token != API_KEY:
        raise HTTPException(
            status_code=401,
            detail={
                "error": {
                    "message": "Invalid or missing API key. Provide header 'Authorization: Bearer YOUR_API_KEY'",
                    "type": "invalid_request_error",
                    "code": "invalid_api_key"
                }
            }
        )
    return True

def format_chat_prompt(messages, enable_thinking=False):
    try:
        meta = llm.metadata
        eos = llm.detokenize([llm.token_eos()]).decode(errors="ignore")
        bos_id = llm.token_bos()
        bos = llm.detokenize([bos_id]).decode(errors="ignore") if bos_id >= 0 else ""
        tmpl = meta.get("tokenizer.chat_template")
        if tmpl:
            formatter = Jinja2ChatFormatter(
                template=tmpl,
                eos_token=eos, bos_token=bos, add_generation_prompt=True,
            )
            return formatter(
                messages=messages, enable_thinking=enable_thinking
            ).prompt
    except Exception as e:
        print(f"Jinja chat template warning: {e}, falling back to Gemma format")

    # Gemma turn markers are used only if the GGUF chat template is unavailable.
    parts = []
    for m in messages:
        role = m.get("role", "user")
        content = m.get("content", "")
        if role == "assistant":
            role = "model"
        parts.append(f"<start_of_turn>{role}\n{content}<end_of_turn>")
    parts.append("<start_of_turn>model\n")
    return "\n".join(parts)

@api_app.get("/health")
def health():
    return {"status": "healthy", "model": MODEL_ID}

@api_app.get("/v1/models")
@api_app.get("/models")
async def list_models(request: Request, authorization: str = Header(None, alias="Authorization"), api_key_header: str = Header(None, alias="api-key"), x_api_key: str = Header(None, alias="x-api-key")):
    verify_api_key(authorization, api_key_header, x_api_key, request.query_params.get("api_key"))
    now = int(time.time())
    model_id = MODEL_ID
    return {
        "object": "list",
        "data": [
            {"id": model_id, "object": "model", "created": now, "owned_by": "kaggle-host", "permission": [], "root": model_id, "parent": None},
            {"id": "default", "object": "model", "created": now, "owned_by": "kaggle-host", "permission": [], "root": model_id, "parent": None}
        ]
    }

In [11]:
# ---------------- Chat completion streaming ----------------
def chat_stream_response(cmpl_id, created_ts, model_name, prompt_tokens, max_tokens, temperature, top_p, stop):
    # Streaming SSE generator with initial keepalive handshake
    def event_stream():
        try:
            # 1. Immediately send initial role chunk to open stream and prevent client timeout
            init_chunk = {
                "id": cmpl_id,
                "object": "chat.completion.chunk",
                "created": created_ts,
                "model": model_name,
                "choices": [{"index": 0, "delta": {"role": "assistant"}, "finish_reason": None}]
            }
            yield f"data: {json.dumps(init_chunk)}\n\n"

            with llm_lock:
                stream_iter = llm.create_completion(
                    prompt=prompt_tokens,
                    max_tokens=max_tokens,
                    temperature=temperature,
                    top_p=top_p,
                    stop=stop,
                    stream=True
                )

                in_think = False
                think_buf = ""

                for chunk in stream_iter:
                    text = chunk["choices"][0].get("text", "")
                    if not text:
                        continue

                    if in_think:
                        think_buf += text
                        if "</think>" in think_buf:
                            remainder = think_buf.split("</think>", 1)[1].lstrip("\n")
                            in_think = False
                            think_buf = ""
                            if remainder:
                                delta_chunk = {
                                    "id": cmpl_id,
                                    "object": "chat.completion.chunk",
                                    "created": created_ts,
                                    "model": model_name,
                                    "choices": [{"index": 0, "delta": {"content": remainder}, "finish_reason": None}]
                                }
                                yield f"data: {json.dumps(delta_chunk)}\n\n"
                        continue

                    if "<think>" in text:
                        pre, post = text.split("<think>", 1)
                        if pre:
                            delta_chunk = {
                                "id": cmpl_id,
                                "object": "chat.completion.chunk",
                                "created": created_ts,
                                "model": model_name,
                                "choices": [{"index": 0, "delta": {"content": pre}, "finish_reason": None}]
                            }
                            yield f"data: {json.dumps(delta_chunk)}\n\n"
                        if "</think>" in post:
                            remainder = post.split("</think>", 1)[1].lstrip("\n")
                            if remainder:
                                delta_chunk = {
                                    "id": cmpl_id,
                                    "object": "chat.completion.chunk",
                                    "created": created_ts,
                                    "model": model_name,
                                    "choices": [{"index": 0, "delta": {"content": remainder}, "finish_reason": None}]
                                }
                                yield f"data: {json.dumps(delta_chunk)}\n\n"
                        else:
                            in_think = True
                            think_buf = post
                        continue

                    # Normal stream token
                    delta_chunk = {
                        "id": cmpl_id,
                        "object": "chat.completion.chunk",
                        "created": created_ts,
                        "model": model_name,
                        "choices": [{"index": 0, "delta": {"content": text}, "finish_reason": None}]
                    }
                    yield f"data: {json.dumps(delta_chunk)}\n\n"

            # Final stop chunk
            stop_chunk = {
                "id": cmpl_id,
                "object": "chat.completion.chunk",
                "created": created_ts,
                "model": model_name,
                "choices": [{"index": 0, "delta": {}, "finish_reason": "stop"}]
            }
            yield f"data: {json.dumps(stop_chunk)}\n\n"

        except Exception as e:
            print("Error in streaming generation:", repr(e))
            traceback.print_exc()
            err_chunk = {
                "id": cmpl_id,
                "object": "chat.completion.chunk",
                "created": created_ts,
                "model": model_name,
                "choices": [{"index": 0, "delta": {"content": f"\n\n[Model Error: {e}]"}, "finish_reason": "stop"}]
            }
            yield f"data: {json.dumps(err_chunk)}\n\n"

        yield "data: [DONE]\n\n"

    return StreamingResponse(
        event_stream(),
        media_type="text/event-stream",
        headers={"Cache-Control": "no-cache", "Connection": "keep-alive", "X-Accel-Buffering": "no"}
    )

In [12]:
@api_app.post("/v1/chat/completions")
@api_app.post("/chat/completions")
async def chat_completions(request: Request, authorization: str = Header(None, alias="Authorization"), api_key_header: str = Header(None, alias="api-key"), x_api_key: str = Header(None, alias="x-api-key")):
    verify_api_key(authorization, api_key_header, x_api_key, request.query_params.get("api_key"))
    body = await request.json()

    raw_messages = body.get("messages", [])
    model_name = body.get("model", MODEL_ID)
    temperature = float(body.get("temperature", 0.6))
    top_p = float(body.get("top_p", 0.9))
    stream = bool(body.get("stream", False))

    stop = body.get("stop", ["<end_of_turn>", "<eos>"])
    if isinstance(stop, str):
        stop = [stop, "<end_of_turn>"]
    elif isinstance(stop, list):
        if "<end_of_turn>" not in stop:
            stop.append("<end_of_turn>")

    # Normalize messages for tokenization
    cleaned_messages = []
    for m in raw_messages:
        c = m.get("content", "")
        if isinstance(c, list):
            parts = []
            for p in c:
                if isinstance(p, dict) and p.get("type") == "text":
                    parts.append(p.get("text", ""))
                elif isinstance(p, str):
                    parts.append(p)
            c = "\n".join(parts)
        cleaned_messages.append({"role": m.get("role", "user"), "content": str(c or "")})

    cmpl_id = f"chatcmpl-{int(time.time()*1000)}"
    created_ts = int(time.time())

    # Tokenize using the proven Jinja2ChatFormatter / ChatML
    prompt_text = format_chat_prompt(cleaned_messages)
    prompt_tokens = llm.tokenize(prompt_text.encode("utf-8"), add_bos=False, special=True)
    prompt_tokens, max_tokens = fit_prompt_to_context(prompt_tokens)

    if not stream:
        with llm_lock:
            res_iter = llm.create_completion(
                prompt=prompt_tokens,
                max_tokens=max_tokens,
                temperature=temperature,
                top_p=top_p,
                stop=stop,
                stream=False
            )
        full_text = res_iter["choices"][0]["text"]
        full_text = re.sub(r"<think>.*?</think>", "", full_text, flags=re.S)
        full_text = re.sub(r"<think>.*$", "", full_text, flags=re.S).strip()
        return JSONResponse({
            "id": cmpl_id,
            "object": "chat.completion",
            "created": created_ts,
            "model": model_name,
            "choices": [{
                "index": 0,
                "message": {"role": "assistant", "content": full_text},
                "finish_reason": "stop"
            }],
            "usage": {
                "prompt_tokens": len(prompt_tokens),
                "completion_tokens": len(llm.tokenize(full_text.encode("utf-8"))),
                "total_tokens": len(prompt_tokens) + len(llm.tokenize(full_text.encode("utf-8")))
            }
        })

    return chat_stream_response(
        cmpl_id=cmpl_id,
        created_ts=created_ts,
        model_name=model_name,
        prompt_tokens=prompt_tokens,
        max_tokens=max_tokens,
        temperature=temperature,
        top_p=top_p,
        stop=stop,
    )

In [13]:
@api_app.post("/v1/completions")
@api_app.post("/completions")
async def text_completions(request: Request, authorization: str = Header(None, alias="Authorization"), api_key_header: str = Header(None, alias="api-key"), x_api_key: str = Header(None, alias="x-api-key")):
    verify_api_key(authorization, api_key_header, x_api_key, request.query_params.get("api_key"))
    body = await request.json()

    prompt = body.get("prompt", "")
    temperature = float(body.get("temperature", 0.6))
    top_p = float(body.get("top_p", 0.9))
    stream = bool(body.get("stream", False))
    prompt_tokens = llm.tokenize(prompt.encode("utf-8"), add_bos=False, special=True)
    prompt_tokens, max_tokens = fit_prompt_to_context(prompt_tokens)

    stop = body.get("stop", ["<end_of_turn>", "<eos>"])
    if isinstance(stop, str):
        stop = [stop, "<end_of_turn>"]
    elif isinstance(stop, list):
        if "<end_of_turn>" not in stop:
            stop.append("<end_of_turn>")

    if not stream:
        with llm_lock:
            res = llm.create_completion(
                prompt=prompt_tokens,
                temperature=temperature,
                top_p=top_p,
                max_tokens=max_tokens,
                stop=stop,
                stream=False
            )
        return JSONResponse(res)

    def stream_prompt():
        with llm_lock:
            for chunk in llm.create_completion(
                prompt=prompt_tokens,
                temperature=temperature,
                top_p=top_p,
                max_tokens=max_tokens,
                stop=stop,
                stream=True
            ):
                yield f"data: {json.dumps(chunk)}\n\n"
        yield "data: [DONE]\n\n"

    return StreamingResponse(
        stream_prompt(),
        media_type="text/event-stream",
        headers={"Cache-Control": "no-cache", "Connection": "keep-alive"}
    )

In [19]:
from fastapi.routing import APIRoute

async def props():
    n_ctx = llm.n_ctx() if callable(llm.n_ctx) else llm.n_ctx
    return {
        "default_generation_settings": {"n_ctx": n_ctx, "params": {}},
        "total_slots": 1,
        "model_path": MODEL_ID,
        "modalities": {"vision": False, "audio": False},
        "chat_template": "",
        "build_info": "colab-llama-cpp-python",
    }

async def tools():
    return []

def add_front(path, fn):
    # drop any earlier registration, then put ours first so Gradio's "/" mount can't swallow it
    api_app.router.routes[:] = [r for r in api_app.router.routes if getattr(r, "path", None) != path]
    api_app.router.routes.insert(0, APIRoute(path, fn, methods=["GET"]))

add_front("/props", props)
add_front("/tools", tools)
print("routes added: /props, /tools")

routes added: /props, /tools


In [20]:
# ---------------- 5. Mount Gradio & Start Background Server ----------------
# Gradio is mounted at root ("/") so the web interface remains accessible at https://<domain>/
api_app = gr.mount_gradio_app(api_app, demo, path="/")

class BackgroundServer(uvicorn.Server):
    def install_signal_handlers(self):
        pass

sock = socket.socket()
sock.bind(("127.0.0.1", 0))
port = sock.getsockname()[1]
sock.close()

print(f"Starting unified FastAPI + Gradio server on port {port}...")

config = uvicorn.Config(app=api_app, host="0.0.0.0", port=port, log_level="warning", loop="asyncio")
uvicorn_server = BackgroundServer(config=config)
server_thread = threading.Thread(target=uvicorn_server.run, daemon=True)
server_thread.start()

# Wait for server readiness
for _ in range(30):
    try:
        with socket.create_connection(("127.0.0.1", port), timeout=2):
            print("Unified server is ready.")
            break
    except OSError:
        time.sleep(1)
else:
    raise RuntimeError("Unified server failed to start.")

Starting unified FastAPI + Gradio server on port 35971...
Unified server is ready.


In [22]:
# ---------------- 6. Connect ngrok tunnel ----------------
try:
    tunnel = ngrok.connect(
        addr=port,
        proto="http",
        url="https://" + ngrok_domain
    )

    print("\n" + "=" * 65)
    print("🚀 SERVER ONLINE - GRADIO UI & OPENAI ENDPOINTS ACTIVE")
    print("=" * 65)
    print(f"🌐 Gradio Web UI        : {tunnel.public_url}")
    print(f"🤖 OpenAI API Base      : {tunnel.public_url}/v1")
    print(f"📦 Model Name in Config : {MODEL_ID}")
    print(f"🔑 API Key Status       : {'Active from Colab Secrets (kept confidential)' if API_KEY else 'No Key Set'}")
    print("=" * 65)
    print(f"""
To configure Continue (in VS Code ~/.continue/config.yaml):

name: Main Config
version: 1.0.0
schema: v1
models:
  - name: Gemma 4 12B Balanced
    provider: openai
    model: {MODEL_ID}
    apiBase: {tunnel.public_url}/v1
    apiKey: {API_KEY}
    roles:
      - chat
      - edit
      - apply
    defaultCompletionOptions:
      contextLength: {MODEL_CONTEXT_TOKENS}
    requestOptions:
      timeout: 300000
      headers:
        ngrok-skip-browser-warning: "1"
""")

except Exception as e:
    print("Ngrok tunnel failed:", repr(e))
    raise


🚀 SERVER ONLINE - GRADIO UI & OPENAI ENDPOINTS ACTIVE
🌐 Gradio Web UI        : https://cherub-yo-yo-daffodil.ngrok-free.dev
🤖 OpenAI API Base      : https://cherub-yo-yo-daffodil.ngrok-free.dev/v1
📦 Model Name in Config : gemma4-12b-balanced
🔑 API Key Status       : Active from Colab Secrets (kept confidential)

To configure Continue (in VS Code ~/.continue/config.yaml):

name: Main Config
version: 1.0.0
schema: v1
models:
  - name: Gemma 4 12B Balanced
    provider: openai
    model: gemma4-12b-balanced
    apiBase: https://cherub-yo-yo-daffodil.ngrok-free.dev/v1
    apiKey: heytherethisismyapikey
    roles:
      - chat
      - edit
      - apply
    defaultCompletionOptions:
      contextLength: 24576
    requestOptions:
      timeout: 300000
      headers:
        ngrok-skip-browser-warning: "1"



In [23]:
import urllib.request, json
# Self-test: local server + public URL
try:
    print("local /health :", urllib.request.urlopen(f"http://127.0.0.1:{port}/health", timeout=10).read().decode())
except Exception as e:
    print("local /health FAILED:", repr(e))
try:
    req = urllib.request.Request(tunnel.public_url + "/health", headers={"ngrok-skip-browser-warning": "1"})
    print("public /health:", urllib.request.urlopen(req, timeout=15).read().decode())
except Exception as e:
    print("public /health FAILED:", repr(e))

local /health : {"status":"healthy","model":"gemma4-12b-balanced"}
public /health: {"status":"healthy","model":"gemma4-12b-balanced"}


In [24]:
import urllib.request, json
base = tunnel.public_url
H = {"ngrok-skip-browser-warning": "1", "Authorization": f"Bearer {API_KEY}", "Content-Type": "application/json"}

def call(path, data=None):
    req = urllib.request.Request(base + path, data=json.dumps(data).encode() if data else None, headers=H)
    try:
        r = urllib.request.urlopen(req, timeout=120)
        return r.status, r.read().decode()[:300]
    except urllib.error.HTTPError as e:
        return e.code, e.read().decode()[:200]
    except Exception as e:
        return "ERR", repr(e)

print("Gradio  /          ", call("/")[0])
print("models  /v1/models ", call("/v1/models"))
print("props   /props     ", call("/props")[0])
print("chat    /v1/chat/completions:")
print(call("/v1/chat/completions", {"model": MODEL_ID, "messages": [{"role": "user", "content": "Say hi in 5 words."}], "max_tokens": 30}))

Gradio  /           200
models  /v1/models  (200, '{"object":"list","data":[{"id":"gemma4-12b-balanced","object":"model","created":1794659240,"owned_by":"kaggle-host","permission":[],"root":"gemma4-12b-balanced","parent":null},{"id":"default","object":"model","created":1794659240,"owned_by":"kaggle-host","permission":[],"root":"gemma4-12b-balanced",')
props   /props      200
chat    /v1/chat/completions:
(200, '{"id":"chatcmpl-1794659240468","object":"chat.completion","created":1794659240,"model":"gemma4-12b-balanced","choices":[{"index":0,"message":{"role":"assistant","content":"Hello, how are you today?"},"finish_reason":"stop"}],"usage":{"prompt_tokens":19,"completion_tokens":8,"total_tokens":27}}')
